# Lesson 5 Demo — Encoder vs Decoder vs Encoder–Decoder (T5)

This notebook demonstrates, side-by-side:

- **Encoder-only** models: produce **embeddings** for similarity / retrieval (not free-form generation)
- **Decoder-only** models (via **Ollama**): produce fluent **next-token generation**, but can hallucinate without grounding
- **Encoder–decoder** models (T5): excel at **structured input → output** tasks (e.g., summarization)

We’ll also run a short “**one fails, one works**” scenario:  
**Decoder-only without evidence** vs **RAG-style grounding (encoder retrieval + decoder)**.

**Decoder model used here:** `llama3:latest` (via local Ollama server)


## 0) Environment setup

You’ll need:
- Ollama running locally (`ollama serve`) and the model pulled (`ollama pull llama3:latest`)
- Python packages: `requests`, `pandas`, `numpy`, `sentence-transformers`, `transformers`, `torch`

If you’re missing packages, install them in your active environment:
```bash
python -m pip install -U requests pandas numpy sentence-transformers transformers torch
```


In [1]:
from __future__ import annotations

import json
import time
import textwrap
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests


### Helper: call Ollama (decoder-only)

We use Ollama’s local HTTP API:
- `POST http://localhost:11434/api/generate`

We’ll default to:
- `temperature`: controls randomness/creativity
- `num_predict`: output length cap


In [8]:
OLLAMA_URL = "http://localhost:11434/api/generate"

def ollama_generate(
    prompt: str,
    model: str = "llama3.1:latest",
    system: Optional[str] = None,
    temperature: float = 0.2,
    top_p: float = 0.9,
    num_predict: int = 200,
) -> str:
    payload: Dict[str, Any] = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": top_p,
            "num_predict": num_predict,
        }
    }
    if system:
        payload["system"] = system

    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=120)
        r.raise_for_status()
        data = r.json()
        return data.get("response", "").strip()
    except Exception as e:
        raise RuntimeError(
            "Ollama call failed. Verify Ollama is running and model is available. "
            f"Original error: {e}"
        )


### Helper: Encoder embeddings + cosine similarity (sentence-transformers)

Encoder-only models shine at:
- similarity search
- clustering
- retrieval/reranking

We’ll use a small, fast embedding model:
- `sentence-transformers/all-MiniLM-L6-v2`


In [9]:
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

_embedder = None

def get_embedder() -> SentenceTransformer:
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer(EMBED_MODEL_NAME)
    return _embedder

def embed_texts(texts: List[str]) -> np.ndarray:
    model = get_embedder()
    embs = model.encode(texts, normalize_embeddings=True)
    return np.array(embs)

def cosine_sim_matrix(embs: np.ndarray) -> np.ndarray:
    # embs are normalized if we used normalize_embeddings=True
    return embs @ embs.T


### Helper: T5 summarization (encoder–decoder)

T5 is a classic encoder–decoder architecture. We’ll use a small checkpoint:

- `t5-small` (fast, good for demo)

We’ll do summarization using:
- `summarize: <text>`


In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

T5_MODEL_NAME = "t5-small"

_t5_tok = None
_t5_model = None

def get_t5():
    global _t5_tok, _t5_model
    if _t5_tok is None or _t5_model is None:
        _t5_tok = AutoTokenizer.from_pretrained(T5_MODEL_NAME)
        _t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_NAME)
    return _t5_tok, _t5_model

def t5_summarize(text: str, max_new_tokens: int = 80) -> str:
    tok, model = get_t5()
    inp = "summarize: " + text.strip().replace("\n", " ")
    enc = tok(inp, return_tensors="pt", truncation=True, max_length=512)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,   # deterministic for comparison
        num_beams=4,
    )
    return tok.decode(out[0], skip_special_tokens=True).strip()


## Demo 1 — Encoder-only: similarity & retrieval (no generation)

We’ll embed a few short “network incident” sentences and compute cosine similarities.

Goal: make it obvious that encoder-only models are **excellent at semantic similarity**, even though they are not used as chat generators.


In [11]:
sentences = [
    "Client failed 802.1X authentication.",
    "EAPOL handshake did not complete.",
    "DNS resolution timed out for api.example.com.",
    "DHCP offer not received by the client.",
    "RADIUS server rejected credentials.",
]

embs = embed_texts(sentences)
sim = cosine_sim_matrix(embs)

df_sim = pd.DataFrame(sim, index=[f"S{i+1}" for i in range(len(sentences))], columns=[f"S{i+1}" for i in range(len(sentences))])
df_sim.round(2)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,S1,S2,S3,S4,S5
S1,1.00,0.18,0.23,0.30,0.45
S2,0.18,1.00,0.17,0.26,0.21
S3,0.23,0.17,1.00,0.16,0.16
S4,0.30,0.26,0.16,1.00,0.22
S5,0.45,0.21,0.16,0.22,1.00


In [12]:
# Show nearest neighbor for each sentence (excluding itself)
for i, s in enumerate(sentences):
    row = sim[i].copy()
    row[i] = -1  # exclude self
    j = int(np.argmax(row))
    print(f"S{i+1}: {s}")
    print(f"  nearest: S{j+1} (score={row[j]:.2f}) -> {sentences[j]}")
    print()


S1: Client failed 802.1X authentication.
  nearest: S5 (score=0.45) -> RADIUS server rejected credentials.

S2: EAPOL handshake did not complete.
  nearest: S4 (score=0.26) -> DHCP offer not received by the client.

S3: DNS resolution timed out for api.example.com.
  nearest: S1 (score=0.23) -> Client failed 802.1X authentication.

S4: DHCP offer not received by the client.
  nearest: S1 (score=0.30) -> Client failed 802.1X authentication.

S5: RADIUS server rejected credentials.
  nearest: S1 (score=0.45) -> Client failed 802.1X authentication.



### Interpretation

Notice how “802.1X authentication”, “EAPOL handshake”, and “RADIUS rejected credentials” cluster together.

That’s the encoder-only strength: **representations** that support search, retrieval, reranking, clustering, classification features, etc.


## Demo 2 — Decoder-only (Ollama / llama3:latest): fluent generation + variability

We’ll ask the same prompt twice:
- higher temperature (more variability)
- lower temperature (more stable)

Goal: show that decoder-only models generate plausible explanations, but may include ungrounded details.


In [13]:
prompt = 'Explain why the network dropped packets. Give 3 likely causes and how to confirm each.'

print('--- Temperature 0.8 (more variability) ---')
print(ollama_generate(prompt, temperature=0.8, num_predict=220))

print('\n--- Temperature 0.2 (more stable) ---')
print(ollama_generate(prompt, temperature=0.2, num_predict=220))


--- Temperature 0.8 (more variability) ---
The frustrating phenomenon of packet drops! Here are three likely causes, along with steps to help you confirm each:

**Cause 1: Congestion**

When a network becomes congested, routers and switches can't forward packets quickly enough, leading to dropped packets.

**How to confirm:**

1. **Check the network utilization**: Use tools like Wireshark or SolarWinds to monitor network traffic and identify periods of high utilization (e.g., above 70%).
2. **Verify router/switch configuration**: Ensure that routers and switches have sufficient buffers, and their buffer sizes are not too small.
3. **Monitor packet loss counters**: Check the router's or switch's counter for packets dropped due to congestion.

**Cause 2: Link Errors**

Link errors occur when data is corrupted during transmission over a physical link, causing packets to be dropped.

**How to confirm:**

1. **Check for error counters**: Use tools like Wireshark or the router/switch's manag

### What to notice

- The answer is usually fluent and plausible.
- But it may “fill in” specifics that were never provided (hallucination risk).
- This is why **grounding** (logs, retrieval, tools) matters in production systems.


## Demo 3 — Encoder–decoder (T5): structured input → output (summarization)

We’ll feed a short incident narrative and ask T5 for a summary.

Goal: show that encoder–decoder models are naturally suited for tasks like **summarization** and other structured transformations.


In [14]:
incident_text = """
A single client reports intermittent connectivity for 3 minutes. 
Logs show repeated DNS timeouts to 10.0.0.53 and a spike in TCP retransmissions. 
The AP reports normal RSSI, but the client roams twice between adjacent APs. 
DHCP renew succeeds, but application traffic experiences periodic stalls.
The issue resolves after the client reconnects.
"""

print('Input (incident narrative):')
print(incident_text.strip())

print('\nT5 summary:')
print(t5_summarize(incident_text, max_new_tokens=60))


Input (incident narrative):
A single client reports intermittent connectivity for 3 minutes. 
Logs show repeated DNS timeouts to 10.0.0.53 and a spike in TCP retransmissions. 
The AP reports normal RSSI, but the client roams twice between adjacent APs. 
DHCP renew succeeds, but application traffic experiences periodic stalls.
The issue resolves after the client reconnects.

T5 summary:


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

logs show repeated DNS timeouts to 10.0.0.53 and a spike in TCP retransmissions. DHCP renew succeeds, but application traffic experiences periodic stalls.


## Demo 4 — “One fails, another works”: Un-grounded decoder vs RAG-style grounding

We’ll create a tiny local corpus of log snippets. Then we ask:

**Question:** “Based on the logs, what is the most likely cause?”

We compare:
1) Decoder-only **without** logs (guessing)
2) **Retrieve top snippets with encoder embeddings**, then prompt the decoder to use **only** that evidence (grounded)


In [15]:
log_snippets = [
    "14:02:11 AP: Deauth reason=4 (Disassociated due to inactivity)",
    "14:02:12 RADIUS: Access-Reject (invalid credentials)",
    "14:02:14 DHCP: No DHCPOFFER received",
    "14:02:15 DNS: timeout to 10.0.0.53",
    "14:02:18 TCP: Retransmissions increased 15%",
    "14:02:20 Client: Roamed from AP1 to AP2 (RSSI -62 dBm)",
    "14:02:22 AP2: Association successful, WPA2 Enterprise",
    "14:02:25 DNS: timeout to 10.0.0.53",
]

question = "Based on the logs, what is the most likely cause of the connectivity issue? Answer in 2-3 sentences."

# (1) Decoder-only without evidence
print('--- Decoder-only (no logs provided) ---')
print(ollama_generate(question, temperature=0.4, num_predict=140))


--- Decoder-only (no logs provided) ---
Unfortunately, I don't have access to any logs or information about a specific connectivity issue. If you'd like to share more context or provide the actual log data, I can try to help you identify the root cause of the problem.


In [16]:
# (2) Encoder retrieval: embed snippets and retrieve top-k relevant to the question
k = 3
texts = [question] + log_snippets
embs_all = embed_texts(texts)
q_emb = embs_all[0]
sn_embs = embs_all[1:]
scores = sn_embs @ q_emb  # cosine since normalized
top_idx = np.argsort(scores)[::-1][:k]

retrieved = [(int(i), float(scores[i]), log_snippets[int(i)]) for i in top_idx]
retrieved


[(4, 0.4140787124633789, '14:02:18 TCP: Retransmissions increased 15%'),
 (7, 0.3859504461288452, '14:02:25 DNS: timeout to 10.0.0.53'),
 (3, 0.37639859318733215, '14:02:15 DNS: timeout to 10.0.0.53')]

In [17]:
evidence_block = "\n".join([f"- {s}" for _, _, s in retrieved])

grounded_prompt = f"""
You are a network assistant. Use ONLY the evidence in the log snippets below.
If evidence is insufficient, say so and list what additional data you would request.

Log snippets:
{evidence_block}

Question: {question}
""".strip()

print('--- Grounded decoder (with retrieved evidence) ---')
print(ollama_generate(grounded_prompt, temperature=0.2, num_predict=170))


--- Grounded decoder (with retrieved evidence) ---
Based on the evidence, it appears that there is a DNS resolution issue causing timeouts to 10.0.0.53. The repeated timeout events (14:02:15 and 14:02:25) suggest a persistent problem with resolving this host. However, I would like to request additional data to determine if this is an isolated issue or part of a larger problem, such as network congestion or a DNS server failure.


### Wrap-up / take away

- **Encoder-only** models: best at embeddings → similarity → retrieval (not free-form generation)
- **Decoder-only** models: best at generation/reasoning → but risk hallucination without grounding
- **Encoder–decoder** models: strong at structured mapping tasks like summarization / translation
- **Agentic / production systems** often combine these: retrieval + generation + tools + constraints
